In [1]:
import os
import sys

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
from dataset_generation.llm import LLM
from dataset_generation.query_generator import QueryGenerator

llm = LLM()  # 로드하는데 10초 조금 넘게 걸림

/home/visuworks2019/miniconda3/envs/dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.12it/s]


In [3]:
query_generator = QueryGenerator(llm=llm)

In [4]:
from data_scraping.common import load_movie_cast, load_movie_data

movie_data = load_movie_data()
cast_data = load_movie_cast()

In [5]:
import pandas as pd

# 모든 열이 보이도록 pandas 옵션 설정
pd.set_option("display.max_columns", None)
movie_data.sample(3)

,movie_id,title,genres,imdb_id,tmdb_id,adult,backdrop_path,id,title_tmdb,original_title,overview,poster_path,media_type,original_language,genre_ids,popularity,release_date,video,vote_average,vote_count,genres_tmdb,language,total_title
848,262633,Pan 3 (1974),,tt2187015,154518.0,False,None,154518,Pan 3,Pan 3,,/hte9AAUFmAlPwbW5o43YGwMucXB.jpg,movie,en,[99],0.1454,1974-02-13,False,5.1,22.0,다큐멘터리,영어,Pan 3
39854,228089,The Haunting at Death Valley Junction (2020),Horror Thriller,tt8359620,752100.0,False,None,752100,The Haunting at Death Valley Junction,The Haunting at Death Valley Junction,,/eSPAZnCvg9byB50l31pSbaUxCGS.jpg,movie,en,"[27, 53]",0.2661,2020-10-06,False,4.8,5.0,공포 스릴러,영어,The Haunting at Death Valley Junction
61608,100304,"Liability, The (2012)",Action Thriller,tt2081437,162145.0,False,/w185C3q1yggd8CE6tiWHAWbgmVD.jpg,162145,라이어빌러티,The Liability,,/4TdsXzigS6xYYO9Qn8oOQrZqX1M.jpg,movie,en,"[53, 35, 80]",2.9897,2012-11-24,False,5.6,204.0,스릴러 코미디 범죄,영어,라이어빌러티 (The Liability)


In [7]:
from dataset_generation.utils import parse_row_to_dict

In [ ]:
from tqdm import tqdm

# 결과를 저장할 리스트
all_queries = []

# 10개 영화에 대해 쿼리 생성
for i in tqdm(range(10), desc="Generating queries"):
    row = movie_data.iloc[i]
    row_dict = parse_row_to_dict(cast_data, row)

    # QueryGenerator로 쿼리 생성
    # queries = query_generator.generate_queries(row_dict)
    queries = query_generator.generate_batchqueries(row_dict)

    # movie_id 가져오기
    movie_id = row["movie_id"]

    # 각 쿼리에 movie_id 추가
    for query_info in queries:
        query_with_id = {
            "movie_id": int(movie_id),
            "query": query_info["query"],
            "query_type": query_info["query_type"],
            "language": query_info["language"],
        }
        all_queries.append(query_with_id)

    # print(f"Movie ID {movie_id}: {len(queries)} queries generated")

print(f"\nTotal queries generated: {len(all_queries)}")

Generating queries: 100%|██████████| 100/100 [00:00<00:00, 1199.92it/s]


## 방법 1: 기본 방식 (리스트에 모아서 한번에 저장)

In [11]:
from dataset_generation.utils import get_queries_file_path, save_queries_to_jsonl

# 파일 경로 생성
jsonl_path = get_queries_file_path(project_root)

# JSONL 형식으로 저장
save_queries_to_jsonl(all_queries, jsonl_path)

Saved 40 queries to: /home/visuworks2019/Projects/mk_folder/movie_recommendation/dataset_generation/data/generated_queries.jsonl


In [12]:
from dataset_generation.utils import load_queries_from_jsonl

# JSONL 파일 로드
df_queries = load_queries_from_jsonl(jsonl_path)

# 데이터 확인
print("\n=== 데이터 형태 ===")
print(f"Shape: {df_queries.shape}")
print("\n=== 쿼리 타입 분포 ===")
print(df_queries["query_type"].value_counts())
print("\n=== 처음 10개 쿼리 ===")
df_queries.head(10)

Loaded 40 queries from: /home/visuworks2019/Projects/mk_folder/movie_recommendation/dataset_generation/data/generated_queries.jsonl

=== 데이터 형태 ===
Shape: (40, 4)

=== 쿼리 타입 분포 ===
query_type
actor     13
hybrid     8
plot       8
mood       7
genre      4
Name: count, dtype: int64

=== 처음 10개 쿼리 ===


,movie_id,query,query_type,language
0,292731,Damián Alcázar 나오는 드라마,actor,ko
1,292731,2022년 드라마,hybrid,ko
2,292731,스페인 드라마,mood,ko
3,292731,Grapa Paola 나오는 드라마,actor,ko
4,104823,연극 배우가 고향으로 돌아가는 영화,plot,ko
5,104823,비브가 연극 교사로 일하는 영화,plot,ko
6,104823,미니 드라이버 나오는 음악 코미디,actor,ko
7,104823,셰익스피어 [템페스트]를 바탕으로 한 록 뮤지컬 영화,mood,ko
8,217503,erin gray 나오는 드라마 영화,actor,ko
9,217503,2020년 드라마 영화,hybrid,ko


## 방법 2: 실시간 저장 방식 (생성할 때마다 바로 추가)

In [ ]:
from tqdm import tqdm

from dataset_generation.utils import append_query_to_jsonl, get_queries_file_path

# 파일 경로 생성 (다른 파일명 사용)
jsonl_path_realtime = get_queries_file_path(project_root, "generated_queries_realtime.jsonl")

# 기존 파일이 있으면 삭제 (새로 시작)
if os.path.exists(jsonl_path_realtime):
    os.remove(jsonl_path_realtime)
    print(f"기존 파일 삭제: {jsonl_path_realtime}\n")

# 통계용 카운터
total_queries = 0

# 10개 영화에 대해 쿼리 생성하고 실시간으로 저장
for i in tqdm(range(10), desc="Generating and saving queries"):
    row = movie_data.iloc[i]
    row_dict = parse_row_to_dict(cast_data, row)

    # QueryGenerator로 쿼리 생성
    queries = query_generator.generate_queries(row_dict)

    # movie_id 가져오기
    movie_id = row["movieId"]

    # 각 쿼리를 생성하자마자 바로 파일에 추가
    for query_info in queries:
        query_with_id = {
            "movie_id": int(movie_id),
            "query": query_info["query"],
            "query_type": query_info["query_type"],
            "language": query_info["language"],
        }
        # 실시간 저장
        append_query_to_jsonl(query_with_id, jsonl_path_realtime)
        total_queries += 1

    print(f"Movie ID {movie_id}: {len(queries)} queries generated and saved")

print(f"\n✅ Total {total_queries} queries saved to: {jsonl_path_realtime}")

In [ ]:
# 실시간 저장된 파일 로드 및 확인
df_queries_realtime = load_queries_from_jsonl(jsonl_path_realtime)

print("\n=== 실시간 저장 vs 일괄 저장 비교 ===")
print(f"일괄 저장: {len(df_queries)} queries")
print(f"실시간 저장: {len(df_queries_realtime)} queries")
print(f"일치 여부: {len(df_queries) == len(df_queries_realtime)}")

df_queries_realtime.head(10)

# 실행 파일!

In [1]:
import os
import sys

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
from dataset_generation.llm import LLM
from dataset_generation.query_generator import QueryGenerator

llm = LLM()  # 로드하는데 10초 조금 넘게 걸림
query_generator = QueryGenerator(llm=llm)

/home/visuworks2019/miniconda3/envs/dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.10it/s]


In [3]:
from data_scraping.common import load_movie_cast, load_movie_data

movie_data = load_movie_data()
cast_data = load_movie_cast()

In [13]:
from tqdm import tqdm

from dataset_generation.utils import append_query_to_jsonl, load_queries_from_jsonl, parse_row_to_dict

df_queries_realtime = load_queries_from_jsonl()
movie_id_list = list(set(movie_data["movie_id"]) - set(df_queries_realtime["movie_id"]))

# 결과를 저장할 리스트
all_queries = []

# 10개 영화에 대해 쿼리 생성
for movie_id in tqdm(movie_id_list, desc="Generating queries"):
    row = movie_data.loc[movie_data["movie_id"] == movie_id].squeeze()
    row_dict = parse_row_to_dict(cast_data, row)

    # QueryGenerator로 쿼리 생성
    queries = query_generator.generate_queries(row_dict)

    # movie_id 가져오기
    movie_id = row["movie_id"]

    # 각 쿼리를 생성하자마자 바로 파일에 추가
    for query_info in queries:
        query_with_id = {
            "movie_id": int(movie_id),
            "query": query_info["query"],
            "query_type": query_info["query_type"],
            "language": query_info["language"],
        }
        # 실시간 저장
        append_query_to_jsonl(query_with_id)

    # print(f"Movie ID {movie_id}: {len(queries)} queries generated")

print(f"\nTotal queries generated: {len(all_queries)}")

Using default path: /home/visuworks2019/Projects/mk_folder/movie_recommendation/dataset_generation/data/generated_queries.jsonl
Loaded 40 queries from: /home/visuworks2019/Projects/mk_folder/movie_recommendation/dataset_generation/data/generated_queries.jsonl


Generating queries:   0%|          | 8/79093 [00:25<71:03:58,  3.23s/it]


KeyboardInterrupt: 